In [1]:
#ddgs

from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
result = search_tool.invoke("What are the olympic medals tally for china and USA in 2024?")
print(result)

/tmp/ipykernel_5956/4192902967.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


२०२६ मे ४ · World map showing nations that have won Summer Olympic medals, as of completion of the 2024 Summer Olympics. World map showing nations that have won Winter ... २०२६ फेब्रुअरी २४ · USA and China tie for most gold medals in the 2024 Summer Olympics · r/news. • 2y ago. USA and China tie for most gold medals in the 2024 Summer Olympics · r/ ...Why does Italy win so many medals at both the Winter and Summer ...USA surpasses 2022 medal count : r/olympics - Redditwww.reddit.com बाट प्राप्त थप परिणामहरू २०२६ फेब्रुअरी १४ · MOST GOLD MEDALS AT EACH SUMMER OLYMPICS BY COUNTRY Global Statistics 2024 - United States / China (tied — 40 gold each, first ever tie) COUNTRIES THAT LED THE ... २०२६ फेब्रुअरी १५ · So they won't get a bump in the medal table as the host country of the Olympics. In one of the medal predictions, China is expected to win only 8 medals, 2 gold ... ४ दिन पहिले · The United States leads by a huge margin with over 1,200 gold medals, followed by the former Soviet Unio

In [2]:
#Shell tool

from langchain_community.tools import ShellTool

shell_tool = ShellTool()

result = shell_tool.invoke("ls -l")
print(result)

Executing command:
 ls -l
total 8
drwxr-xr-x 2 sangam sangam 4096 Jun  7 19:44 __pycache__
-rw-r--r-- 1 sangam sangam 2508 Jun  7 19:49 many_tools.ipynb



/home/sangam/genai/genai/lib/python3.10/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


## Custom Tools


In [3]:
#step1 - Create a function

def multiply(a, b):
    return a * b 

In [4]:
#step2 - add type hints

def multiply(a: int, b: int) -> int:
    return a * b

In [ ]:
#step3 - add tool decorator
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two integers and returns the result."""
    return a * b

In [8]:
result = multiply.invoke({'a': 5, 'b': 10})
print(result)

50


In [9]:
print(multiply.__doc__)

Tool that can operate on any number of inputs.


In [10]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiplies two integers and returns the result.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [11]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiplies two integers and returns the result.', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [12]:
print(search_tool.args_schema.model_json_schema())

{'description': 'Input for the DuckDuckGo search tool.', 'properties': {'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'DDGInput', 'type': 'object'}


## Ways to create tools
- Using @tool decorator
- Using StructuredTool & Pydantic
- Using BasicTool class

In [1]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

In [17]:
class MultiplyInput(BaseModel):
    a: int
    b: int

In [18]:
def multiply_func(input: MultiplyInput) -> int:
    """Multiplies two integers and returns the result."""
    return input.a * input.b

In [19]:
multiply_tool = StructuredTool(
    name="multiply",
    description="Multiplies two integers and returns the result.",
    args_schema=MultiplyInput,
    func=multiply_func
)

In [20]:
print(multiply_tool.args_schema.model_json_schema())

{'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'MultiplyInput', 'type': 'object'}


In [16]:
result = multiply_tool.invoke({'a': 5, 'b': 10})

print(result)

TypeError: multiply_func() got an unexpected keyword argument 'a'

In [21]:
from pydantic import BaseModel
from langchain.tools import StructuredTool

In [22]:
class AddInput(BaseModel):
    x: int
    y: int

In [23]:
def add_func(input: AddInput) -> int:
    """Adds two integers and returns the result."""
    return input.x + input.y

In [26]:
from pydantic import BaseModel
from langchain_core.tools import StructuredTool

# 1. Define the input schema
class AddInput(BaseModel):
    x: int
    y: int

# 2. Function that takes ONE argument (the Pydantic model)
def add_numbers(input: AddInput) -> int:
    """Add two integers."""
    return input.x + input.y

# 3. Create the structured tool
add_tool = StructuredTool(
    name="adder",
    description="Adds two numbers together.",
    args_schema=AddInput,
    func=add_numbers,          # 👈 function with ONE parameter
)

# 4. Invoke with a dictionary
result = add_tool.invoke({"x": 15, "y": 25})
print(result)  # ✅ Output: 40

TypeError: add_numbers() got an unexpected keyword argument 'x'

In [25]:
result = add_tool.invoke({'x': 15, 'y': 25})
print(result)

TypeError: add_func() got an unexpected keyword argument 'x'

In [27]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

# Define the input schema with detailed descriptions for the AI
class CalculatorInput(BaseModel):
    a: float = Field(description="The first number")
    b: float = Field(description="The second number")


In [32]:
def multiply_numbers(a: float,b: float) -> float:
    """Multiply two numbers together."""
    return a * b

In [33]:
# Bind the function and schema together into a tool
multiply_tool = StructuredTool.from_function(
    func=multiply_numbers,
    name="MultiplyCalculator",
    description="Useful for multiplying two numbers together.",
    args_schema=CalculatorInput
)

# You can also run it directly like a normal function:
result = multiply_tool.run({"a": 5.0, "b": 4.5})
print(f"Result: {result}")
# Output: Result: 22.5

Result: 22.5


In [1]:
from langchain.tools import BaseTool
from typing import Type

In [4]:
from pydantic import BaseModel
class MultiplyInput(BaseModel):
    a: int
    b: int

In [5]:
class MultiplyInput(BaseTool):
    name : str = "multiply"
    description : str = "Multiplies two integers and returns the result."
    args_schema : Type[BaseModel] = MultiplyInput
    def _run(self, a: int, b: int) -> int:
        """Multiplies two integers and returns the result."""
        return a * b

In [6]:
multiply_tool = MultiplyInput()

In [7]:
result = multiply_tool.invoke({'a': 5, 'b': 10})
print(result)

50


### Tools kit's


In [8]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Adds two integers and returns the result."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two integers and returns the result."""
    return a * b

In [9]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [10]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(f"Tool Name: {tool.name} = > {tool.description}")

Tool Name: add = > Adds two integers and returns the result.
Tool Name: multiply = > Multiplies two integers and returns the result.
